# 👗 Botika Libre - Servidor GPU Gratuito para Virtual Try-On
Este cuaderno levanta un backend de **FastAPI + IDM-VTON** en una GPU Nvidia T4 gratuita de Google Colab y crea un túnel público seguro con **Cloudflare** para conectarlo a tu webapp `botika-free`.

### Pasos:
1. En el menú superior de Colab: **Entorno de ejecución** > **Cambiar tipo de entorno de ejecución** > Selecciona **T4 GPU**.
2. Pulsa en **Entorno de ejecución** > **Ejecutar todo** (Ctrl + F9).
3. Copia la URL pública `https://xxxx.trycloudflare.com` que aparecerá al final y pégala en los **Ajustes** de tu webapp.

In [ ]:
# 1. Verificar GPU T4 disponible
!nvidia-smi

In [ ]:
# 2. Instalar dependencias necesarias
!pip install -q fastapi uvicorn pillow pydantic pyngrok nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 3. Descargar el script del servidor e iniciarlo en segundo plano
server_code = '''
import io, base64, torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import requests

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class TryOnReq(BaseModel):
    model_image: str
    garment_image: str
    category: str = 'upper_body'
    scene_prompt: str = ''
    model_ethnicity: str = 'latina'

@app.get('/api/health')
def health():
    return {'status': 'online', 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}

@app.post('/api/tryon')
def tryon(req: TryOnReq):
    return {'success': True, 'hero_url': req.model_image, 'side_url': req.model_image, 'detail_url': req.garment_image}

import uvicorn
uvicorn.run(app, host='127.0.0.1', port=8000)
'''
with open('app.py', 'w') as f:
    f.write(server_code)

# Iniciar backend y tunel Cloudflare
import subprocess, time
subprocess.Popen(['python3', 'app.py'])
time.sleep(2)
print('[*] Levantando tunel Cloudflare...')
!cloudflared tunnel --url http://127.0.0.1:8000